# Proyecto LOST: experimentos de Q-Learning

Cada ejecución entrena y testea una configuración, guarda su modelo y agrega una fila a `results/q_learning_experiments.csv`. Los resultados quedan disponibles, en orden de registro, para analizarlos manualmente.

In [ ]:
import json
import re
import time
import uuid
from datetime import datetime
from html import escape
from pathlib import Path

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display

from experiment_logger import (
    EXPERIMENT_COLUMNS,
    OVERNIGHT_SUMMARY_COLUMNS,
    SEARCH_RUN_COLUMNS,
    append_experiment_result,
    load_experiment_results,
    load_overnight_summary,
    load_search_run_results,
    summarize_test_results,
    summarize_training_history,
)
from q_learning_agent import QLearningAgent

ENV_ID = "MountainCarContinuous-v0"
CSV_PATH = Path("results/q_learning_experiments.csv")
SEARCH_RUNS_CSV_PATH = Path("results/q_learning_search_runs.csv")
OVERNIGHT_SUMMARY_CSV_PATH = Path("results/q_learning_overnight_summary.csv")

## Ejecución y registro

`run_experiment(config, seed=0, notes="")` usa `episodes`, `max_steps` y `test_episodes` como parámetros de la corrida. El resto de la configuración corresponde al agente. El shaping potencial solo modifica la actualización Q durante entrenamiento; el test y sus métricas siempre usan la recompensa original del ambiente.

In [ ]:
def run_experiment(config, seed=0, notes=""):
    config = dict(config)
    if "config_name" not in config:
        raise ValueError("config debe incluir config_name.")

    config_name = str(config.pop("config_name"))
    episodes = int(config.pop("episodes", 50))
    max_steps = int(config.pop("max_steps", 999))
    test_episodes = int(config.pop("test_episodes", 10))
    verbose = bool(config.pop("verbose", False))
    progress_interval = int(config.pop("progress_interval", 500))
    epsilon_start = float(config.pop("epsilon_start", config.pop("epsilon", 1.0)))

    agent_config = {
        "position_bins": 40,
        "velocity_bins": 40,
        "action_bins": 9,
        "alpha": 0.1,
        "gamma": 0.99,
        "epsilon": epsilon_start,
        "epsilon_min": 0.05,
        "epsilon_decay": 0.995,
        "reward_shaping": "none",
        "shaping_position_weight": 1.0,
        "shaping_velocity_weight": 0.5,
        "q_init": 0.0,
        "explicit_action_values": None,
        "seed": seed,
    }
    configurable_keys = set(agent_config) - {"epsilon", "seed"}
    unknown_keys = set(config) - configurable_keys
    if unknown_keys:
        raise ValueError(f"Parámetros desconocidos: {unknown_keys}")
    agent_config.update(config)

    experiment_id = uuid.uuid4().hex[:12]
    safe_name = re.sub(r"[^a-zA-Z0-9_-]+", "_", config_name).strip("_")
    safe_name = safe_name or "experiment"
    model_path = (
        Path("models")
        / f"q_learning_{safe_name}_seed{seed}_{experiment_id}.pkl"
    )

    env = gym.make(ENV_ID, render_mode="rgb_array")
    agent = QLearningAgent(**agent_config)
    try:
        start_time = time.perf_counter()
        history = agent.train_agent(
            env, episodes=episodes, max_steps=max_steps,
            verbose=verbose, progress_interval=progress_interval,
        )
        training_time = time.perf_counter() - start_time
        test_results = agent.test_agent(
            env, episodes=test_episodes, max_steps=max_steps
        )
    finally:
        env.close()

    agent.save(model_path, metadata={
        "experiment_id": experiment_id, "config_name": config_name,
        "seed": seed, "episodes": episodes, "notes": notes,
    })
    result_row = {
        "experiment_id": experiment_id,
        "timestamp": datetime.now().astimezone().isoformat(timespec="seconds"),
        "config_name": config_name,
        "seed": seed,
        "position_bins": agent_config["position_bins"],
        "velocity_bins": agent_config["velocity_bins"],
        "action_bins": agent.action_bins,
        "alpha": agent_config["alpha"],
        "gamma": agent_config["gamma"],
        "epsilon_start": epsilon_start,
        "epsilon_min": agent_config["epsilon_min"],
        "epsilon_decay": agent_config["epsilon_decay"],
        "episodes": episodes,
        "max_steps": max_steps,
        **summarize_training_history(history),
        "test_episodes": test_episodes,
        **summarize_test_results(test_results),
        "training_time_seconds": round(training_time, 3),
        "model_path": model_path.as_posix(),
        "notes": notes,
        "reward_shaping": agent.reward_shaping,
        "shaping_position_weight": agent.shaping_position_weight,
        "shaping_velocity_weight": agent.shaping_velocity_weight,
        "q_init": agent.q_init,
        "explicit_action_values": (
            json.dumps(agent.explicit_action_values)
            if agent.explicit_action_values is not None else ""
        ),
        "final_epsilon": agent.epsilon,
    }
    result_row = append_experiment_result(result_row, CSV_PATH)
    print(f"Experimento {experiment_id} registrado en {CSV_PATH}")
    return agent, history, test_results, result_row

## Tres configuraciones de ejemplo

Se usan 20 episodios para una validación rápida. Para una corrida real basta cambiar `EXPERIMENT_EPISODES` a 2000, 3000 o 5000 antes de ejecutar esta celda. Cada nueva ejecución agrega filas; nunca reemplaza las anteriores.

In [ ]:
EXPERIMENT_EPISODES = 20
TEST_EPISODES = 2

baseline_simple = {
    "config_name": "baseline_simple",
    "position_bins": 30,
    "velocity_bins": 30,
    "action_bins": 7,
    "alpha": 0.10,
    "gamma": 0.99,
    "epsilon_start": 1.0,
    "epsilon_min": 0.05,
    "epsilon_decay": 0.995,
    "episodes": EXPERIMENT_EPISODES,
    "max_steps": 999,
    "test_episodes": TEST_EPISODES,
}

balanced = {
    "config_name": "balanced",
    "position_bins": 40,
    "velocity_bins": 40,
    "action_bins": 9,
    "alpha": 0.15,
    "gamma": 0.99,
    "epsilon_start": 1.0,
    "epsilon_min": 0.05,
    "epsilon_decay": 0.995,
    "episodes": EXPERIMENT_EPISODES,
    "max_steps": 999,
    "test_episodes": TEST_EPISODES,
}

fine_grained = {
    "config_name": "fine_grained",
    "position_bins": 50,
    "velocity_bins": 50,
    "action_bins": 11,
    "alpha": 0.10,
    "gamma": 0.995,
    "epsilon_start": 1.0,
    "epsilon_min": 0.05,
    "epsilon_decay": 0.997,
    "episodes": EXPERIMENT_EPISODES,
    "max_steps": 999,
    "test_episodes": TEST_EPISODES,
}

In [ ]:
RUN_EXAMPLE_EXPERIMENTS = False

if RUN_EXAMPLE_EXPERIMENTS:
    baseline_run = run_experiment(
        baseline_simple, seed=10, notes="Baseline de grilla compacta"
    )
    balanced_run = run_experiment(
        balanced, seed=20, notes="Mayor resolución y alpha"
    )
    fine_grained_run = run_experiment(
        fine_grained, seed=30, notes="Grilla fina y decay más lento"
    )
else:
    print("Ejemplos omitidos. Cambiá RUN_EXAMPLE_EXPERIMENTS a True para entrenarlos.")

## Comparación manual

Primero se muestra la lista de búsquedas. Editá `COMPARISON_SEARCH_ID` para decidir cuál analizar. La vista inicial usa la corrida overnight completa de 60 experimentos.

In [ ]:
def display_results_table(rows, columns=None):
    columns = columns or EXPERIMENT_COLUMNS
    if not rows:
        display(HTML("<p>No hay experimentos registrados.</p>"))
        return

    header = "".join(f"<th>{escape(column)}</th>" for column in columns)
    body = "".join(
        "<tr>"
        + "".join(f"<td>{escape(str(row.get(column, '')))}</td>" for column in columns)
        + "</tr>"
        for row in rows
    )
    display(HTML(f"<div style='overflow-x:auto'><table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table></div>"))

In [ ]:
experiment_rows = load_experiment_results(CSV_PATH)
search_run_rows = load_search_run_results(SEARCH_RUNS_CSV_PATH)
overnight_summary_rows = load_overnight_summary(OVERNIGHT_SUMMARY_CSV_PATH)
print(f"Experimentos registrados: {len(experiment_rows)}")

In [ ]:
search_overview_columns = [
    "search_id", "search_name", "profile", "status",
    "number_of_configs_completed", "episodes", "test_episodes",
    "seeds", "total_search_time_seconds",
]
display(HTML("<h3>Búsquedas disponibles</h3>"))
display_results_table(search_run_rows, search_overview_columns)

# EDITÁ ESTE VALOR para comparar otra búsqueda.
COMPARISON_SEARCH_ID = "6b358d688579"

comparison_columns = [
    "search_id",
    "profile",
    "search_name",
    "config_name",
    "seed",
    "position_bins",
    "velocity_bins",
    "action_bins",
    "alpha",
    "gamma",
    "epsilon_decay",
    "final_epsilon",
    "reward_shaping",
    "q_init",
    "explicit_action_values",
    "episodes",
    "train_last_100_avg_env_reward",
    "train_last_100_avg_learning_reward",
    "train_last_100_max_position",
    "test_success_rate",
    "test_success_count",
    "test_avg_reward",
    "test_avg_steps",
    "test_avg_max_position",
    "training_time_seconds",
    "model_path",
    "notes",
    "run_key",
]
aggregate_columns = [
    "config_name", "seeds_completed",
    "mean_test_success_rate", "std_test_success_rate",
    "mean_test_avg_reward", "std_test_avg_reward",
    "mean_test_avg_steps", "std_test_avg_steps",
    "mean_test_avg_max_position", "mean_training_time_seconds",
    "mean_train_last_100_avg_env_reward",
    "mean_train_success_rate_last_100",
]
comparison_experiment_rows = [
    row for row in experiment_rows if row["search_id"] == COMPARISON_SEARCH_ID
]
comparison_summary_rows = [
    row for row in overnight_summary_rows if row["search_id"] == COMPARISON_SEARCH_ID
]
display(HTML(f"<h3>Comparación elegida: {escape(COMPARISON_SEARCH_ID)}</h3>"))
if comparison_summary_rows:
    display_results_table(comparison_summary_rows, aggregate_columns)
else:
    display_results_table(comparison_experiment_rows, comparison_columns)

## Gráficos descriptivos

Para búsquedas con varias seeds se grafican medias y desvíos del resumen agregado. Para otras búsquedas se muestran las corridas individuales. El orden es el del CSV.

In [ ]:
def _numeric(rows, column):
    return np.array([float(row[column]) for row in rows], dtype=float)


def _horizontal_comparison(rows, metrics, title):
    if not rows:
        print("No hay resultados para el search_id elegido.")
        return
    labels = [row["config_name"] for row in rows]
    y = np.arange(len(labels))
    fig, axes = plt.subplots(2, 3, figsize=(18, 15))
    for axis, (value_column, error_column, metric_title) in zip(axes.flat, metrics):
        values = _numeric(rows, value_column)
        errors = _numeric(rows, error_column) if error_column else None
        bars = axis.barh(y, values, xerr=errors, alpha=0.82, capsize=3)
        axis.set_yticks(y, labels)
        axis.invert_yaxis()
        axis.set_title(metric_title)
        axis.grid(axis="x", alpha=0.25)
        axis.bar_label(bars, fmt="%.2f", padding=3, fontsize=8)
    fig.suptitle(title, fontsize=15)
    fig.tight_layout()
    plt.show()


if comparison_summary_rows:
    aggregate_metrics = [
        ("mean_test_success_rate", "std_test_success_rate", "Éxito de test: media ± desvío (%)"),
        ("mean_test_avg_reward", "std_test_avg_reward", "Reward original de test: media ± desvío"),
        ("mean_test_avg_steps", "std_test_avg_steps", "Pasos de test: media ± desvío"),
        ("mean_test_avg_max_position", None, "Máxima posición media de test"),
        ("mean_training_time_seconds", None, "Tiempo medio de entrenamiento (s)"),
        ("mean_train_success_rate_last_100", None, "Éxito medio en últimos 100 episodios (%)"),
    ]
    _horizontal_comparison(
        comparison_summary_rows, aggregate_metrics,
        f"Resumen por configuración — search_id={COMPARISON_SEARCH_ID}",
    )
else:
    individual_metrics = [
        ("test_success_rate", None, "Éxito de test (%)"),
        ("test_avg_reward", None, "Reward original de test"),
        ("test_avg_steps", None, "Pasos medios de test"),
        ("test_avg_max_position", None, "Máxima posición media de test"),
        ("training_time_seconds", None, "Tiempo de entrenamiento (s)"),
        ("train_success_rate_last_100", None, "Éxito en últimos 100 episodios (%)"),
    ]
    _horizontal_comparison(
        comparison_experiment_rows, individual_metrics,
        f"Corridas individuales — search_id={COMPARISON_SEARCH_ID}",
    )

## Elección manual de una configuración

Después de mirar la tabla y los gráficos, escribí exactamente un `config_name` en `CHOSEN_CONFIG_NAME`. La celda mostrará sus resultados por seed y las rutas de sus modelos; no toma ninguna decisión por vos.

In [ ]:
available_config_names = list(dict.fromkeys(
    row["config_name"] for row in comparison_experiment_rows
))
print("Configuraciones disponibles:")
for name in available_config_names:
    print(" -", name)

# EDITÁ ESTE VALOR después de comparar las métricas.
CHOSEN_CONFIG_NAME = ""

if CHOSEN_CONFIG_NAME:
    chosen_rows = [
        row for row in comparison_experiment_rows
        if row["config_name"] == CHOSEN_CONFIG_NAME
    ]
    if not chosen_rows:
        raise ValueError("CHOSEN_CONFIG_NAME no pertenece a la búsqueda elegida.")
    chosen_columns = [
        "config_name", "seed", "test_success_rate",
        "test_avg_reward", "test_std_reward", "test_avg_steps",
        "test_avg_max_position", "train_success_rate_last_100",
        "reward_shaping", "q_init", "model_path",
    ]
    display_results_table(chosen_rows, chosen_columns)
else:
    print("Todavía no elegiste una configuración. Editá CHOSEN_CONFIG_NAME.")

## Resultados de búsquedas de hiperparámetros

Esta sección opcional vuelve a cargar los CSV y los presenta en su orden de registro para compararlos manualmente. El resumen overnight contiene agregados descriptivos por configuración y búsqueda.

In [ ]:
SHOW_COMPLETE_HISTORY = False

if SHOW_COMPLETE_HISTORY:
    display(HTML("<h3>Historial completo de búsquedas</h3>"))
    display_results_table(search_run_rows, SEARCH_RUN_COLUMNS)
    display(HTML("<h3>Historial completo de experimentos</h3>"))
    display_results_table(experiment_rows, comparison_columns)
    display(HTML("<h3>Resumen overnight completo</h3>"))
    display_results_table(overnight_summary_rows, OVERNIGHT_SUMMARY_COLUMNS)
else:
    print("Historial completo oculto. Cambiá SHOW_COMPLETE_HISTORY a True para verlo.")